<a href="https://colab.research.google.com/github/av-jones/DimABSA/blob/main/AJ_PhD_Week1_RDoC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PhD Week 1: RDoC Framework

**Prepare data with RDoC features for Week 2**

This cell concatenates the train_aug and valid_aug datasets into a single list called full_aug. It then writes each item of full_aug as a JSON line to a file named rdoc_full.jsonl in the /content/ directory. Finally, it prints a confirmation message indicating the number of saved pairs.

In [ ]:
from google.colab import drive
#drive.mount('/content/drive')
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt

def load_jsonl(fp):
    return [json.loads(l) for l in open(fp) if l.strip()]

train_rest = load_jsonl('/content/sample_data/eng_restaurant_small_train.jsonl')
train_laptop = load_jsonl('/content/sample_data/eng_laptop_small_train.jsonl')
valid_rest = load_jsonl('/content/sample_data/eng_restaurant_valid.jsonl')
valid_laptop = load_jsonl('/content/sample_data/eng_laptop_valid.jsonl')

RDoc Class

This class is designed to compute 'positive' and 'negative' sentiment scores based on predefined keywords.
It has two class attributes: POS (Positive) and NEG (Negative), which are #dictionaries categorizing sentiment-related words. POS contains words for  'reward', while NEG contains words for 'threat' and 'loss'.
The @classmethod compute(cls, text) takes a text string as input. It converts the text to lowercase and then counts the occurrences of words from the POS and NEG dictionaries within the text. It returns a dictionary with pos (total positive word count) and neg (total negative word count).

Augment Function

This function takes raw data (data) and a domain (e.g., 'restaurant', 'laptop') as input and transforms it into an augmented format suitable for further processing.
It iterates through each item in the input data.
For each item, it first calls RDoC.compute() on the item['Text'] to get the rdoc_pos and rdoc_neg scores.
It then iterates through each Quadruplet within the item. A quadruplet typically contains an Aspect, a VA (sentiment value), and a Category.
For each quadruplet, it extracts the aspect (using Aspect if not 'NULL', otherwise Category), the va, and includes the previously computed rdoc_pos, rdoc_neg scores, and the domain.
It compiles these augmented items into a list, aug, and returns it.'''

In [ ]:
class RDoC:
    POS = {'reward': ['love', 'excellent', 'amazing', 'wonderful', 'great', 'best', 'delicious']}
    NEG = {'threat': ['terrible', 'horrible', 'awful', 'worst'], 'loss': ['disappointed', 'poor', 'bad']}
    @classmethod
    def compute(cls, text):
        t = text.lower()
        return {'pos': sum(sum(1 for w in words if w in t) for words in cls.POS.values()),
                'neg': sum(sum(1 for w in words if w in t) for words in cls.NEG.values())}

def augment(data, domain):
    aug = []
    for item in data:
        rdoc = RDoC.compute(item['Text'])
        for quad in item['Quadruplet']:
            aspect = quad['Aspect'] if quad['Aspect'] != 'NULL' else quad['Category']
            aug.append({'text': item['Text'], 'aspect': aspect, 'va': quad['VA'],
                       'rdoc_pos': rdoc['pos'], 'rdoc_neg': rdoc['neg'], 'domain': domain})
    return aug

train_aug = augment(train_rest, 'restaurant') + augment(train_laptop, 'laptop')
valid_aug = augment(valid_rest, 'restaurant') + augment(valid_laptop, 'laptop')
print(f'Train: {len(train_aug)}, Valid: {len(valid_aug)}')

Train: 7540, Valid: 1892


In [ ]:
# Save for Week 2
full_aug = train_aug + valid_aug
with open('/content/sample_data/rdoc_full.jsonl', 'w') as f:
    for item in full_aug:
        f.write(json.dumps(item) + '\n')
print(f'✓ Saved {len(full_aug)} RDoC-augmented pairs')

✓ Saved 9432 RDoC-augmented pairs
